# Business Entity Resolution — full-scale GPU training run

This notebook runs the **same pipeline that scored 0.97 on the leaderboard (v2)**, with the laptop's
RAM/time compromises removed, plus a small number of targeted fixes from error analysis. It does **not**
change the modeling approach (blocking → LightGBM 2-stage → expected-F0.5 set selection) — that approach
is proven. It removes the constraints that were only there because v2 was trained on a 16 GB laptop:

| Laptop (v2, scored 0.97) | This notebook |
|---|---|
| 30% of (country, state) partitions for training (`--final` default) | **100%** of partitions |
| Training pairs capped at 40M regardless of RAM (hardcoded) | Cap now scales with detected RAM (raised to a 2B ceiling) |
| 3 folds, lr 0.05, 3000 max rounds | Configurable below — defaults to more rounds / lower lr, since compute is no longer the constraint |
| Blocking top-k tuned tight to keep candidate count low | Widened (`eda/outputs/02_blocking_recall.txt` showed real recall headroom at larger k) |

**New since the 0.97 run** (see the accompanying diagnosis for why):
1. `hn_digit_perm` feature — flags house numbers that are the same digits in a different order (e.g. `35`
   vs `53`), a corruption pattern the existing house-number features (`hn_ham1`, `hn_absdiff`, ...) don't catch.
2. Blocking `K_*` widened and the prune loosened (`eda/outputs/14_blocking_misses.txt` and
   `02_blocking_recall.txt` both showed uncaptured recall at larger k — this was tuned tight only to keep
   candidate volume laptop-sized).
3. Isotonic recalibration of stage-2 scores, fit on OOF and kept only if it wins on the dense
   (test-density-simulated) validation view. This matters specifically for the `expf` decision rule, which
   sums *actual probability mass* (not just rank) to pick each S1's optimal match set — including the
   all-important singleton (empty-set) case, which is binary in the score (1.0 or 0.0).

**Honest expectation-setting:** 0.97 → 0.99+ is a large jump. Full data + more rounds should recover some
of the underfitting HANDOFF.md flagged (stage 1 hit its round cap every time on the laptop). The new
feature and calibration are targeted, evidence-based, but unproven at scale — that's exactly what the OOF
report in Step 4 below is for: **it is your gate**. Do not spend a leaderboard submission until you've seen
that number and it beats 0.97-ish OOF. This local OOF has tracked the real leaderboard score closely in this
project (v1: 0.974 OOF vs 0.956 LB; v2: OOF close to 0.97 LB) so it's a trustworthy proxy, not a guess.

**Compliance reminders (do not change):** vectorizers/surface-form dictionaries are fit on **train only**
(never test) — this is already enforced in the code (`assert split == "train"` in `block.py`). Final model
is LightGBM (MIT-licensed, classical ML, no parameter-count concern under the ≤8B rule). Do not add any
external data/lookups.


## Step 0 — Setup

Upload this whole `code/business_entity_resolution/` folder (with the edited `src/ber/` package) to the
GPU box, and upload/mount the dataset (`dataset/train/*.tsv`, `dataset/test/*.tsv`) somewhere on disk.
Edit the paths in **Step 1 (Config)** below to match.


In [ ]:
import sys, subprocess
# Core deps (CPU). LightGBM GPU support is attempted separately below (optional, best-effort).
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "polars==1.44.2", "pyarrow==25.0.1", "numpy", "scipy",
                "scikit-learn==1.9.1", "sparse-dot-topn==1.2.0", "rapidfuzz==3.14.6",
                "lightgbm==4.7.0", "Unidecode==1.4.0", "psutil"], check=True)
print("done")


### Optional — GPU-accelerated LightGBM

This is genuinely optional and skippable. Gradient-boosted trees on tabular data are usually CPU-bound in
a different way than deep learning is GPU-bound — LightGBM's CPU histogram training is already fast and
highly parallel, and the standard PyPI wheel has **no CUDA support built in** (it must be compiled from
source against your CUDA toolkit, which is fragile to do inside a notebook and easy to get wrong). The real
value of your 80GB-GPU box for *this* pipeline is the RAM and CPU cores that come with a machine like that
(removing the laptop's 16GB ceiling), not the GPU FLOPs themselves.

If you still want to try `device_type="cuda"`, the cell below attempts a source build and **falls back to
the standard CPU wheel on any failure** — safe to run, but expect it to take several minutes and possibly
fail depending on your CUDA toolkit setup. If it fails, just proceed; nothing downstream depends on it.

If you want the GPU's actual compute put to use for a bigger score jump later, the natural next step (not
in this notebook — this one deliberately sticks to the safer, proven approach) is a fine-tuned transformer
cross-encoder ensembled with this model. Ask if you want that version too.


In [ ]:
import subprocess, sys
try:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-binary", "lightgbm",
                    "--config-settings=cmake.define.USE_CUDA=ON", "lightgbm"], check=True, timeout=900)
    import lightgbm as lgb, numpy as np
    d = lgb.Dataset(np.random.rand(1000, 5), label=np.random.randint(0, 2, 1000))
    lgb.train({"objective": "binary", "device_type": "cuda", "verbose": -1}, d, 5)
    GPU_LGBM = True
    print("CUDA LightGBM build OK")
except Exception as e:
    GPU_LGBM = False
    print(f"CUDA LightGBM not available ({e!r}) -- continuing on CPU, which is expected/fine.")


## Step 1 — Config

Edit these paths. `WORK` needs ~50-100+ GB free disk for a full-scale run (feature memmaps, checkpoints,
blocking candidate parquets per partition) — more than the laptop's ~30GB because we're no longer dropping
partitions to fit RAM.


In [ ]:
import os, sys, time, json, psutil

CODE_SRC   = "/path/to/code/business_entity_resolution/src"   # folder containing run_all.py and ber/
TRAIN_DIR  = "/path/to/dataset/train"
TEST_DIR   = "/path/to/dataset/test"
WORK       = "/path/to/work"      # scratch: parquet, checkpoints, models, memmaps
OUT_DIR    = "/path/to/output"    # final matching_results.tsv + candidate_pairs.tsv land here

sys.path.insert(0, CODE_SRC)
os.makedirs(WORK, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)

ram_gb = psutil.virtual_memory().total / 2**30
print(f"RAM: {ram_gb:.0f} GB total, {psutil.virtual_memory().available/2**30:.0f} GB available | CPUs: {os.cpu_count()}")
print(f"disk free at WORK: {__import__('shutil').disk_usage(WORK).free/2**30:.0f} GB")


## Step 2 — Full-scale training settings

`BER_MAX_PAIRS` is left unset on purpose so it auto-derives from your detected RAM (see
`train._auto_max_pairs`, raised from a hard 40M ceiling to scale with `psutil`-detected RAM, capped at 2B
as a sanity bound). Override it explicitly below if you want to force a specific number.

Rounds/lr/patience are pushed harder than the laptop's `--final` defaults (3000/0.05/100) since
HANDOFF.md's own diagnosis was that stage 1 **hit the round cap every time** (underfit, not overfit) —
early stopping will still cut it short once validation loss stops improving, this just removes the
artificial ceiling. Folds are kept at 3 (matches the proven v1/v2 setup) rather than raised to 5, to keep
wall-clock bounded — raise `BER_FOLDS` if you have time budget to spare; each fold trains sequentially.


In [ ]:
os.environ["PYTHONUTF8"] = "1"   # Windows: Indic text breaks the default cp1252 encoding; harmless elsewhere
os.environ["BER_FOLDS"]    = "3"
os.environ["BER_ROUNDS"]   = "4000"
os.environ["BER_LR"]       = "0.04"
os.environ["BER_PATIENCE"] = "150"
# os.environ["BER_MAX_PAIRS"] = "200000000"   # uncomment to override the auto RAM-based cap
# os.environ["BER_THREADS"]   = "32"            # uncomment to override CPU-count autodetect for blocking

DEV_FRAC = 1.0   # 1.0 = ALL (country, state) partitions for training (vs 0.3 on the laptop's --final)

from ber import io_raw, prepare, block, train as ber_train, predict
import importlib
for m in (io_raw, prepare, block, ber_train, predict):
    importlib.reload(m)
ber_train.ROUNDS = int(os.environ["BER_ROUNDS"]); ber_train.PATIENCE = int(os.environ["BER_PATIENCE"])
ber_train.PARAMS["learning_rate"] = float(os.environ["BER_LR"])
print("settings applied")


## Step 3 — Parquet + prepare

TSV → Parquet, then normalisation (state/region parsing, India transliteration dictionary and street-form
surface forms — all **learned from TRAIN only**, exactly as before). A few minutes.


In [ ]:
RAW = os.path.join(WORK, "raw")
t0 = time.time()
if not os.path.exists(os.path.join(RAW, "train_links.parquet")):
    io_raw.tsv_to_parquet(TRAIN_DIR, TEST_DIR, RAW)
print(f"parquet done [{time.time()-t0:.0f}s]")


In [ ]:
t0 = time.time()
if not os.path.exists(os.path.join(WORK, "train_s1.parquet")):
    prepare.main(RAW, WORK)
print(f"prepare done [{time.time()-t0:.0f}s]")


## Step 4 — Block + train on the FULL training set

This is the big one. `block.main(..., frac=1.0)` builds candidates over **all** partitions (no dev-world
resampling needed at `frac=1.0` — that resampling logic only exists to fix a train/production mismatch
that occurs when subsetting partitions; at full scale there's no subsetting, so it's naturally skipped).

Both steps are internally checkpointed per-partition (`WORK/blk_train_<country>/<partition>.parquet`) and
per-stage (`tr_p1.npy`, `models_stage1.pkl`) — safe to interrupt and rerun the same cell to resume.

**This is your validation gate.** Read the printed OOF F0.5 (stage1 / stage2 / stage2-calibrated, by
country) before going any further. Compare it to the known 0.97 leaderboard result. If it's clearly
better, continue to test prediction below. If it's *not* clearly better, stop and tell me the numbers —
something in the full-scale run or my changes needs looking at before you spend a submission on it.


In [ ]:
t0 = time.time()
block.main(WORK, "train", ["US", "India"], DEV_FRAC)
print(f"block_train done [{time.time()-t0:.0f}s]")


In [ ]:
t0 = time.time()
ber_train.main(WORK, RAW, ["US", "India"])
print(f"train done [{time.time()-t0:.0f}s]")

rep = json.load(open(os.path.join(WORK, "train_report.json"), encoding="utf-8"))
print(f"\nOOF F0.5 -- stage1: {rep['f1']:.4f}  stage2: {rep['f2']:.4f}  stage2+calibration: {rep['f2cal']:.4f}")
print(f"stage2 by country: {rep['by_country2']}")
print(f"stage2+cal by country: {rep['by_country2cal']}")
print("\n(France is not in this OOF -- train has no French labels. This number covers US+India only,")
print(" same as every OOF number in HANDOFF.md. Judge France via a leaderboard France A/B, not this.)")


## Step 5 — Block + predict on the full test set (US, India, France)

Same candidate generation, run over the test set including France (which has no training labels, but the
blocking/normalisation logic already has explicit France handling learned label-free from S1's own
addresses — see `text.py:learn_fr_city`, `fr_region`).


In [ ]:
t0 = time.time()
block.main(WORK, "test", ["US", "India", "France"], 1.0)
print(f"block_test done [{time.time()-t0:.0f}s]")


In [ ]:
t0 = time.time()
predict.main(WORK, OUT_DIR)
print(f"predict done [{time.time()-t0:.0f}s]")


## Step 6 — Validate the submission format

Runs the official validator (no external deps) before you spend a leaderboard submission on a malformed
file.


In [ ]:
import subprocess
validator = os.path.join(os.path.dirname(CODE_SRC), "..", "..", "utils", "validate_submission.py")
validator = os.path.abspath(validator)
if not os.path.exists(validator):
    print(f"validator not found at {validator} -- point this at utils/validate_submission.py from student_resource/")
else:
    r = subprocess.run([sys.executable, validator,
                         "--matching", os.path.join(OUT_DIR, "matching_results.tsv"),
                         "--candidate", os.path.join(OUT_DIR, "candidate_pairs.tsv"),
                         "--test-dir", TEST_DIR], capture_output=True, text=True)
    print(r.stdout); print(r.stderr)


## Step 7 — Next steps

- If the validator prints `PASS`: upload `matching_results.tsv` to the leaderboard.
- Keep `candidate_pairs.tsv` and this whole `WORK` folder — they're required in the final submission zip
  (`output/`, `code/business_entity_resolution/`) alongside the methodology doc.
- **France A/B** (from HANDOFF.md, still the best way to measure France in isolation): re-score saved test
  predictions with a different decision rule in ~2 minutes, no retraining, via
  `predict.decide_only(WORK, OUT_DIR2, decision=("expf", 0.3, "uniq"))` — submit two files differing only
  in the French rows to isolate France's contribution to the leaderboard score.
- If the OOF gate in Step 4 looked good but you still want more headroom toward 0.99, the next lever (not
  in this notebook) is a fine-tuned transformer cross-encoder ensembled with this model's score — that
  would actually use your GPU's compute, unlike this tree-based pipeline. Ask if you want that built.
